# Mecanum Hybrid SMO / MPC / Kalman-PID / Fuzzy Controller

Companion simulation notebook implementing the hybrid, run-time-selectable
controller architecture specified in
`instructions/mecanum-hybrid-smo-mpc-pid-fuzzy-notebook.md`.

**Plant:** the existing roller-kinematics + dynamic-LuGre bristle friction plant
(reused verbatim from `run_one.jl:675-742`), but converted to a **torque/voltage
input** plant: the ASMC controller is stripped out of the ODE RHS and replaced by
a zero-order-hold (ZOH) motor-voltage bus, updated by scheduled discrete
callbacks (SMO/Kalman estimator, ASMC, MPC, Kalman-PID, fuzzy supervisor, mixer).

**Hard constraints carried through:**
- Only IMU (accel + gyro) and wheel encoders are measured -- absolute pose is
  never measured, only dead-reckoned (drifts).
- Actuation is **motor voltage** through a youBot DC-motor + gearbox model
  (`youbot-base.cfg` parameters); voltage/current saturation is physical.

**Note on data logging:** `datastore.jl`'s `compute_labels`/`assemble_dataframe`
are hard-coded to the single-controller 39-D ASMC+DOB state and the
`asmc_torques`/`asmc_torques_vel` call signature (`(u, t, params, asmc, eso)`).
The hybrid architecture has a materially different state layout (30/34-D plant,
no in-state ASMC gains) and a multi-controller bus, so this notebook defines its
own lightweight `log_run`/`save_run` pair (Sec 4.12) that writes an **additive**
Arrow schema next to the existing one, rather than forcing an incompatible reuse
of `DataStore.compute_labels`. `profiles.jl` (`Profiles`) is reused unchanged.


## 1. Imports

In [ ]:
using LinearAlgebra
using OrdinaryDiffEq
using DiffEqCallbacks
using StaticArrays
using DataFrames
using Arrow
using Plots
using Printf
using ProgressMeter
using Random
using OSQP
using SparseArrays

Random.seed!(0)

include("profiles.jl");  using .Profiles
println("Profiles: ", join(sort(collect(keys(Profiles.BUILDERS))), ", "))


## 2. `HybridConfig` -- the only parametrization cell (Sec 4.10)

Single source of truth for which architectures are active and at what rate.
Tagged `parameters` per project convention (mirrors the main simulator notebook's
Cell 2).

In [ ]:
Base.@kwdef struct HybridConfig
    # --- tracking / estimator selection ---
    tracking::Symbol        = :velocity      # :pose | :velocity
    estimator::Symbol       = :kalman        # :kalman | :smo | :none
    use_dhat::Bool          = false          # SMO disturbance feedforward; requires estimator==:smo

    # --- controller selection ---
    use_asmc::Bool          = true
    use_mpc::Bool           = true
    use_pid::Bool           = true
    fuzzy::Bool              = true
    fixed_weights::NTuple{3,Float64} = (1/3, 1/3, 1/3)   # used when fuzzy=false

    # --- rates (Hz) ---
    f_est::Float64           = 1000.0
    f_asmc::Float64          = 1000.0
    f_mpc::Float64            = 100.0
    f_pid::Float64            = 100.0
    f_fuzzy::Float64          = 50.0
    f_mix::Float64            = 1000.0

    # --- motor / plant variant ---
    dynamic_electrical::Bool = false          # false => 30-D quasi-static; true => 34-D w/ motor currents

    # --- sensor / solver knobs ---
    seed::Int                = 0
    reltol::Float64          = 1e-8
    dtmax::Float64           = 1e-3
end

# --- Stand-in default HybridConfig used by the definition cells below and the
# demo run at the bottom of the notebook. The selection-matrix cell (Sec 11)
# constructs many HybridConfig instances directly and does not depend on this.
cfg = HybridConfig()


## 3. Plant structs -- `PlatformParams`, `LuGreParams`, `MotorParams` (Sec 4.3, 4.4b)

`PlatformParams` and `LuGreParams` are reused **unchanged** from `run_one.jl`
(struct definitions + the `PlatformParams(base; mu_friction)` constructor).
`MotorParams` is new: the youBot DC-motor + gearbox actuator map (Sec 3 params
table, Sec 4.4b).

In [ ]:
struct PlatformParams
    # Geometry
    h::Float64          # half-length (m)
    l::Float64          # half-width (m)
    R::Float64          # wheel outer radius (m)
    Ra::Float64         # roller axle distance (m)
    # Masses/inertias
    m::Float64
    m_wheel::Float64
    J_wheel::Float64
    J_roller::Float64
    ms::Float64         # total mass
    Is::Float64         # platform moment of inertia
    # Viscous friction cases
    p1_case1::Float64
    p2_case1::Float64
    p1_case2::Float64
    p2_case2::Float64
    # Friction
    f_coulomb::Float64
    N_total::Float64
    rollers_per_wheel::Int
    # Per-wheel static fields (SVector for stack allocation)
    delta::SVector{4,Float64}       # roller axis angle
    wc_x::SVector{4,Float64}        # wheel center X
    wc_y::SVector{4,Float64}        # wheel center Y
    aX::Float64                     # COM offset X
    aY::Float64                     # COM offset Y
    N_per_roller::SVector{4,Float64}
    M_inv::SMatrix{3,3,Float64,9}        # plant body mass matrix (no wheel reflection)
    M_aug::SMatrix{3,3,Float64,9}        # augmented mass matrix (with wheel reflection)
    M_aug_inv::SMatrix{3,3,Float64,9}    # inverse of the augmented mass matrix
    Max_torque::Float64
end

function PlatformParams(base::AbstractDict; mu_friction::Float64)
    geo = base["platform"]["geometry"];  mas = base["platform"]["mass"]
    com = base["platform"]["com_offset"]; vis = base["platform"]["viscous"]
    con = base["platform"]["contact"]
    h  = geo["h"];  l = geo["l"];  R = geo["R"];  Ra = geo["Ra"]
    m  = mas["m"];  m_wheel = mas["m_wheel"]
    J_wheel = mas["J_wheel"];  J_roller = mas["J_roller"]
    ms = m + 4.0 * m_wheel
    Is = mas["Is"]
    p1_case1 = vis["p1_case1"]; p2_case1 = vis["p2_case1"]
    p1_case2 = vis["p1_case2"]; p2_case2 = vis["p2_case2"]
    f_coulomb = mu_friction
    N_total = m * 9.81
    rollers_per_wheel = con["rollers_per_wheel"]
    delta = SVector(-pi/4, pi/4, pi/4, -pi/4)
    wc_x  = SVector(h, h, -h, -h);  wc_y = SVector(l, -l, l, -l)
    aX = com["aX"];  aY = com["aY"]

    N_per_roller = SVector(
        N_total/4 * (1 + aX/h + aY/l) + m_wheel * 9.81,
        N_total/4 * (1 + aX/h - aY/l) + m_wheel * 9.81,
        N_total/4 * (1 - aX/h + aY/l) + m_wheel * 9.81,
        N_total/4 * (1 - aX/h - aY/l) + m_wheel * 9.81
    )
    M_mat = @SMatrix [ ms       0.0     -m*aY;
                       0.0      ms       m*aX;
                      -m*aY     m*aX     Is   ]
    M_inv = inv(M_mat)
    m_tilde   = ms + 4 * J_wheel / R^2
    I_psi_aug = Is + 4 * (l + h)^2 / R^2 * J_wheel
    M_aug = @SMatrix [ m_tilde   0.0       -m*aY ;
                       0.0       m_tilde    m*aX ;
                      -m*aY      m*aX       I_psi_aug ]
    M_aug_inv = inv(M_aug)
    Max_torque = 10.0
    PlatformParams(h, l, R, Ra, m, m_wheel, J_wheel, J_roller, ms, Is,
                   p1_case1, p2_case1, p1_case2, p2_case2, f_coulomb,
                   N_total, rollers_per_wheel, delta, wc_x, wc_y,
                   aX, aY, N_per_roller, M_inv, M_aug, M_aug_inv, Max_torque)
end

Base.@kwdef struct LuGreParams
    sigma0::Float64   = 1.64e3
    sigma1::Float64   = 1.6
    sigma2::Float64   = 0.0
    sigma0_s::Float64 = 1.09e3
    sigma1_s::Float64 = 1.1
    sigma2_s::Float64 = 0.0
    stiction_ratio::Float64 = 1.1
    v_str::Float64    = 0.01
    w_str::Float64    = 0.01
    use_mindlin::Bool = true
    mindlin_iters::Int = 2
    eps_reg::Float64  = 1e-4
end

@inline stribeck_g(s, mu_c, ratio, vs) = mu_c * (1 + (ratio - 1) * exp(-(s / vs)^2))
coupling_of(fm::Symbol) = fm === :lugre_adamov ? :adamov : :uncoupled

# DYNAMIC LuGre: forces/torque + bristle derivatives -- copied verbatim from run_one.jl.
@inline function lugre_dyn_rates(lg::LuGreParams, coupling::Symbol,
                                 f::Real, N::Real, chi::Real,
                                 w_z::Real, Vpx::Real, Vpy::Real,
                                 zx::Real, zy::Real, zs::Real)
    er  = lg.eps_reg
    Vp  = sqrt(Vpx^2 + Vpy^2 + er^2)
    awz = sqrt(w_z^2 + er^2)
    c_t = (8/(3*pi)) * awz * chi

    if coupling === :adamov
        if lg.use_mindlin
            znorm = sqrt(zx^2 + zy^2 + 1e-18)
            dstar = (lg.stiction_ratio * f) / lg.sigma0
            sfrac = clamp(znorm / dstar, 0.0, 1.0)
            b     = max(1 - sfrac, 1e-9)
            fsl   = 1 - b^(2/3)
        else
            fsl = 1.0
        end
        s_t = fsl * c_t + Vp
        s_s = (16/(3*pi)) * awz * chi + 5 * Vp
    else
        s_t = Vp
        s_s = (16/(3*pi)) * awz * chi + er
    end

    g_t = stribeck_g(s_t, f, lg.stiction_ratio, lg.v_str)
    g_s = stribeck_g(s_s, f, lg.stiction_ratio, lg.w_str)

    dzx = Vpx - lg.sigma0   * s_t / g_t * zx
    dzy = Vpy - lg.sigma0   * s_t / g_t * zy
    dzs = w_z - lg.sigma0_s * s_s / g_s * zs

    Fx = -N * (lg.sigma0   * zx + lg.sigma1   * dzx + lg.sigma2   * Vpx)
    Fy = -N * (lg.sigma0   * zy + lg.sigma1   * dzy + lg.sigma2   * Vpy)
    Mz = -N * chi^2 * (lg.sigma0_s * zs + lg.sigma1_s * dzs + lg.sigma2_s * w_z)
    return Fx, Fy, Mz, dzx, dzy, dzs
end

lugre = LuGreParams()

# --- Smoothed sawtooth (tanh-peak version, run_one.jl:212-218) ---
const TANH_K = 60.0
@inline function sawtooth_tanh(theta)
    s = sin(12*theta)
    c = cos(12*theta)
    return atan(TANH_K * s, TANH_K * c + 1) / 12
end
const sawtooth_approx = sawtooth_tanh

@inline smooth_sat(M, L, n::Int = 3) = M / (1 + (M / L)^(2n))^(1 / (2n))

# ---------------------------------------------------------------------------
# MotorParams -- youBot DC-motor + gearbox actuator (Sec 3, Sec 4.4b).
# Defaults from youbot_driver/config/youbot-base.cfg where available; the
# electrical pair (Ra, La) and V_max are calibratable/verify-only stand-ins.
# ---------------------------------------------------------------------------
Base.@kwdef struct MotorParams
    Kt::Float64    = 0.0335         # torque constant [N*m/A]
    G::Float64     = 9405/364       # gear reduction (wheel/motor); motor:wheel ~25.84:1
    Kb::Float64    = 0.0335         # back-EMF constant [V*s/rad] (ideal-motor SI equality with Kt)
    Ra::Float64    = 2.0            # winding resistance [Ohm] -- CALIBRATE
    La::Float64    = 1e-4           # winding inductance [H]  -- CALIBRATE (quasi-static default: small)
    i_max::Float64 = 5.0            # current limit [A]        -- CALIBRATE
    i_comm::Float64 = 0.2           # commutation current [A] (cfg: CommutationCurrent)
    V_max::Float64 = 24.0           # bus voltage limit [V]    -- verify against youBot battery
    eta::Float64    = 0.9           # gear efficiency
    tau_f::Float64  = 0.01          # motor-shaft Coulomb friction [N*m] (paper-typical)
    cpr::Int        = 4000          # encoder ticks/round on the MOTOR shaft
    dynamic_electrical::Bool = false
end


## 4. `ControllerBus`, `PlantODEParams` (Sec 4.2, 4.9)

The mutable rendezvous shared between the ODE RHS and the discrete callbacks.
`bus.v_cmd` is the ZOH input the plant reads; every callback writes one of the
other fields.

In [ ]:
mutable struct ControllerBus
    v_cmd::SVector{4,Float64}          # ZOH motor-voltage input to the plant
    x_hat::SVector{6,Float64}          # [Vx, Vy, psi_dot, psi, Xo, Yo] estimated state
    d_hat::SVector{3,Float64}          # SMO lumped-disturbance estimate (zero if unused)
    W_asmc::SVector{3,Float64}
    W_mpc::SVector{3,Float64}
    W_pid::SVector{3,Float64}
    weights::SVector{3,Float64}        # (w_asmc, w_mpc, w_pid), normalized over ENABLED controllers
    K_asmc::SVector{3,Float64}         # ASMC adaptive gains [Kx, Ky, Kpsi]
    pid_integral::SVector{3,Float64}
    mpc_warm::Vector{Float64}          # OSQP warm-start vector (flattened decision variables)
    mpc_last::SVector{4,Float64}       # last MPC voltage solution (fallback on infeasible)
    tau_wheel_applied::SVector{4,Float64}   # diagnostic: last wheel torque actually applied
    y_meas::NamedTuple                 # measurement cache (theta, omega, a_x, a_y, g_z)
end

function ControllerBus()
    ControllerBus(
        SVector(0.0,0.0,0.0,0.0),
        SVector(0.0,0.0,0.0,0.0,0.0,0.0),
        SVector(0.0,0.0,0.0),
        SVector(0.0,0.0,0.0), SVector(0.0,0.0,0.0), SVector(0.0,0.0,0.0),
        SVector(1/3,1/3,1/3),
        SVector(6.0,6.0,24.0),
        SVector(0.0,0.0,0.0),
        zeros(4*10),   # generous flat warm-start buffer for a Np~10 horizon
        SVector(0.0,0.0,0.0,0.0),
        SVector(0.0,0.0,0.0,0.0),
        (theta=SVector(0.0,0.0,0.0,0.0), omega=SVector(0.0,0.0,0.0,0.0),
         a_x=0.0, a_y=0.0, g_z=0.0),
    )
end

struct PlantODEParams
    params::PlatformParams
    chi::Float64
    p1::Float64
    p2::Float64
    coupling::Symbol
    lugre::LuGreParams
    motor::MotorParams
    bus::ControllerBus
end


## 5. `motor_torque` + `plant_rhs!` (Sec 4.1, 4.4b)

`motor_torque` is the youBot DC-motor + gearbox map: commanded voltage + wheel
speed -> applied wheel torque. `plant_rhs!` is the roller-kinematics + LuGre
plant (copied verbatim from `run_one.jl:675-742`) with the controller stripped
out: wheel torque now comes from `motor_torque(bus.v_cmd, ...)` instead of
`ctrl(...)`.

State layout -- **30-D quasi-static** (default, `La ~ 0`, no motor-current
states):

| idx | meaning | idx | meaning |
|---|---|---|---|
| 1-3 | Vx, Vy, psi_dot | 17-18 | Xo, Yo (world position, eval-only) |
| 4 | psi | 19-22 | zx (bristle) |
| 5-8 | theta_i (encoder truth) | 23-26 | zy (bristle) |
| 9-12 | omega_i (encoder truth) | 27-30 | zs (spin bristle) |
| 13-16 | gamma_i (roller spin) | | |

The **34-D full-electrical** variant (`cfg.dynamic_electrical=true`) appends
`[31:34] = i1..i4` motor currents with `La*di/dt = V - Ra*i - Kb*G*omega`; not
implemented in this notebook's RHS (kept as a documented extension point -- see
`motor_torque`'s `dynamic_electrical` branch, which already reads a state
current when given one).

In [ ]:
@inline function motor_torque(v_cmd::AbstractVector, omega::AbstractVector, motor::MotorParams)
    G, Kt, Kb, Ra, eta, tau_f, i_max = motor.G, motor.Kt, motor.Kb, motor.Ra, motor.eta, motor.tau_f, motor.i_max
    omega_m = G .* omega
    # quasi-static current: i = clamp((V - Kb*omega_m)/Ra, +-i_max), smoothed via tanh (C-infinity,
    # consistent with the project's other smoothed saturations) instead of a hard clamp.
    i_raw = (v_cmd .- Kb .* omega_m) ./ Ra
    i = smooth_sat.(i_raw, i_max, 3)
    tau_f_smooth = tau_f .* tanh.(20.0 .* omega_m)     # C-infinity replacement for tau_f*sign(omega_m)
    tau_m = Kt .* i .- tau_f_smooth
    tau_wheel = G * eta .* tau_m
    return SVector{4,Float64}(tau_wheel), SVector{4,Float64}(i)
end

# Full-electrical variant: torque directly from a state current (no algebraic solve).
@inline function motor_torque_from_current(i::AbstractVector, omega::AbstractVector, motor::MotorParams)
    omega_m = motor.G .* omega
    tau_f_smooth = motor.tau_f .* tanh.(20.0 .* omega_m)
    tau_m = motor.Kt .* i .- tau_f_smooth
    return SVector{4,Float64}(motor.G * motor.eta .* tau_m)
end

# ---------------------------------------------------------------------------
# plant_rhs!(du, u, p, t) -- 30-D quasi-static torque/voltage-input plant.
# Physics (roller contact velocities, LuGre calls, dv/dwi/dgi) is copied
# verbatim from run_one.jl:675-742; the only change is Mi_sat now comes from
# motor_torque(bus.v_cmd, wi, motor) instead of ctrl(...).
# ---------------------------------------------------------------------------
function plant_rhs!(du, u, p::PlantODEParams, t)
    params, chi, p1, p2, coupling, lugre, motor, bus = p.params, p.chi, p.p1, p.p2, p.coupling, p.lugre, p.motor, p.bus

    Vx, Vy, psi_dot, psi = u[1], u[2], u[3], u[4]
    ti = SVector(u[5],  u[6],  u[7],  u[8])
    wi = SVector(u[9],  u[10], u[11], u[12])
    gi = SVector(u[13], u[14], u[15], u[16])
    zx = SVector(u[19], u[20], u[21], u[22])
    zy = SVector(u[23], u[24], u[25], u[26])
    zs = SVector(u[27], u[28], u[29], u[30])

    px, py = params.wc_x, params.wc_y
    R, Rd  = params.R, params.Ra
    delta  = params.delta
    ms     = params.ms

    sdi = sin.(delta); cdi = cos.(delta); tdi = tan.(delta)
    ti_t  = sawtooth_approx.(ti)
    sti_t = sin.(ti_t);  cti_t = cos.(ti_t);  tti_t = tan.(ti_t)
    DYi   = Rd .* tdi .* tti_t

    Vpi_x = @. Vx - psi_dot * (py + DYi) - wi * R +
               gi * sdi * (Rd * cti_t - R) + DYi * gi * cdi * sti_t
    Vpi_y = @. Vy + psi_dot * px +
               gi * cdi * (R * cti_t - Rd)
    wzi   = @. psi_dot - gi * (-sti_t * cdi)

    Ni = params.N_per_roller
    fr = ntuple(4) do i
        lugre_dyn_rates(lugre, coupling, params.f_coulomb, Ni[i], chi,
                        wzi[i], Vpi_x[i], Vpi_y[i], zx[i], zy[i], zs[i])
    end
    Fx_i = SVector(fr[1][1], fr[2][1], fr[3][1], fr[4][1])
    Fy_i = SVector(fr[1][2], fr[2][2], fr[3][2], fr[4][2])
    Mz_i = SVector(fr[1][3], fr[2][3], fr[3][3], fr[4][3])
    dzx  = SVector(fr[1][4], fr[2][4], fr[3][4], fr[4][4])
    dzy  = SVector(fr[1][5], fr[2][5], fr[3][5], fr[4][5])
    dzs  = SVector(fr[1][6], fr[2][6], fr[3][6], fr[4][6])

    # Applied wheel torque from the ZOH motor-voltage bus (replaces ctrl(...)).
    Mi_sat, i_motor = motor_torque(bus.v_cmd, wi, motor)
    bus.tau_wheel_applied = Mi_sat     # diagnostic only; safe (mutable field write, single writer=RHS)

    Mz_rolleri = @. px * Fy_i - py * Fx_i + Mz_i
    RHS0 = sum(Fx_i)          + ms * psi_dot * Vy  + params.m * params.aX * psi_dot^2
    RHS1 = sum(Fy_i)          - ms * psi_dot * Vx  + params.m * params.aY * psi_dot^2
    RHS2 = sum(Mz_rolleri)    - params.m * psi_dot * (params.aX * Vx + params.aY * Vy)
    dv = params.M_inv * SVector(RHS0, RHS1, RHS2)

    dwi = @. (Mi_sat - Fx_i * R - p1 * wi) / params.J_wheel
    dgi = @. (- p2 * gi
              - Fx_i * (sdi * (R - Rd * cti_t) - DYi * sti_t * cdi)
              - Fy_i * cdi * (Rd - R * cti_t)
              - Mz_i * sti_t * cdi) / params.J_roller / params.rollers_per_wheel

    du[1] = dv[1];  du[2] = dv[2];  du[3] = dv[3]
    du[4] = psi_dot
    @inbounds for i in 1:4
        du[4+i]  = wi[i]
        du[8+i]  = dwi[i]
        du[12+i] = dgi[i]
    end
    du[17] = Vx * cos(psi) - Vy * sin(psi)
    du[18] = Vx * sin(psi) + Vy * cos(psi)
    @inbounds for i in 1:4
        du[18+i] = dzx[i]   # u[19:22]
        du[22+i] = dzy[i]   # u[23:26]
        du[26+i] = dzs[i]   # u[27:30]
    end
    return nothing
end

function build_initial_state(cfg::HybridConfig)
    n = cfg.dynamic_electrical ? 34 : 30
    u0 = zeros(n)
    ref = Profiles.active_ref()
    u0[1] = 0.0; u0[2] = 0.0; u0[3] = 0.0
    u0[4] = ref.psi(0.0)
    u0[5:8]  .= 0.1
    u0[9:12] .= 0.0
    u0[13:16].= 0.0
    return u0
end

const ABSTOL_HYBRID = vcat(
    fill(1.0e-8, 3), fill(1.0e-7, 1),      # body vel, psi
    fill(1.0e-8, 4), fill(1.0e-7, 4),      # wtheta, womega
    fill(1.0e-7, 4),                       # gamma
    fill(1.0e-7, 2),                       # world pos
    fill(1.0e-10, 8), fill(1.0e-7, 4),     # bristle x/y, bristle-rot
)   # 3+1+4+4+4+2+8+4 = 30


## 6. `SensorModel` + `simulate_measurement` (Sec 4.4)

Maps true plant state to a realistic **IMU + encoder** measurement. This is the
only information any controller/estimator may use -- true pose (Xo, Yo, psi) and
true Vx, Vy are forbidden as controller inputs and are only read by `log_run`
for evaluation.

In [ ]:
Base.@kwdef struct SensorModel
    enc_cpr::Int      = 4000 * round(Int, 9405/364)   # ticks/rev, wheel-referred (motor cpr * G)
    sigma_omega::Float64 = 0.01     # encoder-derived wheel speed noise [rad/s]
    sigma_acc::Float64   = 0.05     # accelerometer noise [m/s^2]
    sigma_gyro::Float64  = 0.01     # gyro noise [rad/s]
    gyro_bias_rw::Float64 = 1e-4    # gyro random-walk bias rate [rad/s per sqrt(s)]
    acc_bias::Float64     = 0.02    # constant accel bias [m/s^2]
    seed::Int              = 0
end

mutable struct SensorState
    rng::Random.AbstractRNG
    gyro_bias::Float64
end
SensorState(sm::SensorModel) = SensorState(Random.Xoshiro(sm.seed), 0.0)

# simulate_measurement(u, du, sm, sstate, t) -> y::NamedTuple
#
# y = (theta, omega, a_x, a_y, g_z), with additive noise, gyro bias random-walk,
# and encoder quantization. NEVER exposes true pose/body velocity.
#   a_x = Vx_dot - psi_dot*Vy ,  a_y = Vy_dot + psi_dot*Vx ,  g_z = psi_dot
# `du` must be the RHS evaluated at `u,t` (accel feedforward comes from the same
# plant physics -- not finite-differenced, per Sec 9's "do NOT finite-difference").
function simulate_measurement(u::AbstractVector, du::AbstractVector, sm::SensorModel, sstate::SensorState, dt::Real)
    theta_true = SVector(u[5], u[6], u[7], u[8])
    omega_true = SVector(u[9], u[10], u[11], u[12])
    Vx, Vy, psi_dot = u[1], u[2], u[3]
    Vx_dot, Vy_dot = du[1], du[2]

    tick = 2*pi / sm.enc_cpr
    theta_meas = round.(theta_true ./ tick) .* tick
    omega_meas = omega_true .+ sm.sigma_omega .* randn(sstate.rng, 4)

    sstate.gyro_bias += sm.gyro_bias_rw * sqrt(max(dt, 1e-9)) * randn(sstate.rng)
    g_z = psi_dot + sstate.gyro_bias + sm.sigma_gyro * randn(sstate.rng)

    a_x = Vx_dot - psi_dot * Vy + sm.acc_bias + sm.sigma_acc * randn(sstate.rng)
    a_y = Vy_dot + psi_dot * Vx + sm.acc_bias + sm.sigma_acc * randn(sstate.rng)

    return (theta = theta_meas, omega = omega_meas, a_x = a_x, a_y = a_y, g_z = g_z)
end


## 7. Estimators -- `KalmanEstimator` (default) and `SMOEstimator` (optional) (Sec 4.5)

Fuse encoder-derived body velocity + IMU into an estimated body state; pose is
dead-reckoned and therefore drifts (unobservable -- expected, not a bug).

In [ ]:
# Body-frame velocity pseudo-measurement: invert the O-config wheel map
# (algebraic analog of the forward map used in plant_rhs!/asmc_torques).
@inline function wheel_omega_to_body_vel(omega::AbstractVector, params::PlatformParams)
    R, l, h = params.R, params.l, params.h
    lever = R / (l + h)
    # forward map (no-slip, small-angle): w1..w4 <- [Vx,Vy,psi_dot] via the O-config Jacobian;
    # here we use its least-squares pseudo-inverse (4 measurements -> 3 states).
    J = @SMatrix [ 1.0/R  -1.0/R  -lever/R;
                   1.0/R   1.0/R   lever/R;
                   1.0/R   1.0/R  -lever/R;
                   1.0/R  -1.0/R   lever/R]
    x = (J' * J) \ (J' * omega)
    return SVector{3,Float64}(x)
end

Base.@kwdef mutable struct KalmanEstimator
    Qn::SMatrix{3,3,Float64,9} = SMatrix{3,3,Float64,9}(Diagonal(SVector(1e-2, 1e-2, 1e-2)))
    Rn::SMatrix{3,3,Float64,9} = SMatrix{3,3,Float64,9}(Diagonal(SVector(1e-2, 1e-2, 1e-3)))
    P::SMatrix{3,3,Float64,9}  = SMatrix{3,3,Float64,9}(Diagonal(SVector(1.0,1.0,1.0)))
end

Base.@kwdef mutable struct SMOEstimator
    L::SVector{3,Float64} = SVector(15.0, 15.0, 15.0)     # sliding gain
    K::SVector{3,Float64} = SVector(5.0, 5.0, 5.0)         # integral (d-hat) gain
    delta::Float64        = 1e-2                            # smooth-switching boundary layer
    zeta::SVector{3,Float64} = SVector(0.0, 0.0, 0.0)       # internal integral state (-> d_hat)
end

# estimator_update!(bus, y, est, params, dt)
#
# Advance the estimator one tick; fuse y -> bus.x_hat = [Vx,Vy,psi_dot,psi,Xo,Yo].
# Pose block is dead-reckoned (unobservable -> growing covariance, expected).
# `est` is KalmanEstimator or SMOEstimator (dispatch on type).
function estimator_update!(bus::ControllerBus, y::NamedTuple, est::KalmanEstimator, params::PlatformParams, dt::Real)
    v_meas = wheel_omega_to_body_vel(y.omega, params)
    v_meas = SVector(v_meas[1], v_meas[2], y.g_z)     # yaw rate from gyro, not the wheel pseudo-inverse

    x_prev = bus.x_hat
    v_prev = SVector(x_prev[1], x_prev[2], x_prev[3])
    a_ff   = SVector(y.a_x, y.a_y, 0.0)
    v_pred = v_prev .+ dt .* a_ff
    P_pred = est.P + est.Qn

    H = SMatrix{3,3,Float64,9}(I)
    S = H * P_pred * H' + est.Rn
    Kgain = P_pred * H' * inv(S)
    innov = v_meas .- H * v_pred
    v_upd = v_pred .+ Kgain * innov
    est.P = (SMatrix{3,3,Float64,9}(I) - Kgain * H) * P_pred

    psi_upd = x_prev[4] + dt * v_upd[3]
    c, s = cos(psi_upd), sin(psi_upd)
    Xo_upd = x_prev[5] + dt * (v_upd[1]*c - v_upd[2]*s)
    Yo_upd = x_prev[6] + dt * (v_upd[1]*s + v_upd[2]*c)

    bus.x_hat = SVector(v_upd[1], v_upd[2], v_upd[3], psi_upd, Xo_upd, Yo_upd)
    return nothing
end

function estimator_update!(bus::ControllerBus, y::NamedTuple, est::SMOEstimator, params::PlatformParams, dt::Real)
    v_meas = wheel_omega_to_body_vel(y.omega, params)
    v_meas = SVector(v_meas[1], v_meas[2], y.g_z)

    x_prev = bus.x_hat
    v_hat  = SVector(x_prev[1], x_prev[2], x_prev[3])
    a_ff   = SVector(y.a_x, y.a_y, 0.0)

    e = v_meas .- v_hat
    switch = e ./ sqrt.(e.^2 .+ est.delta^2)     # smooth sign(e)

    v_dot = a_ff .+ est.zeta .+ est.L .* switch
    zeta_dot = est.K .* switch

    v_hat_new  = v_hat .+ dt .* v_dot
    est.zeta   = est.zeta .+ dt .* zeta_dot
    bus.d_hat  = est.zeta

    psi_upd = x_prev[4] + dt * v_hat_new[3]
    c, s = cos(psi_upd), sin(psi_upd)
    Xo_upd = x_prev[5] + dt * (v_hat_new[1]*c - v_hat_new[2]*s)
    Yo_upd = x_prev[6] + dt * (v_hat_new[1]*s + v_hat_new[2]*c)

    bus.x_hat = SVector(v_hat_new[1], v_hat_new[2], v_hat_new[3], psi_upd, Xo_upd, Yo_upd)
    return nothing
end


## 8. Control laws -- ASMC, MPC, Kalman-PID (Sec 4.6)

In [ ]:
Base.@kwdef struct ASMCParams
    gamma_x::Float64     = 8.0
    gamma_y::Float64     = 15.0
    gamma_psi::Float64   = 25.0
    eps::Float64         = 0.0175
    eps_psi::Float64     = 0.08
    K_max_x::Float64     = 60.0
    K_max_y::Float64     = 80.0
    K_max_psi::Float64   = 100.0
    lam_x_min::Float64   = 0.1
    lam_x_max::Float64   = 1.5
    lam_y_min::Float64   = 0.2
    lam_y_max::Float64   = 2.5
    lam_psi_min::Float64 = 0.5
    lam_psi_max::Float64 = 5.0
    mu_xy::Float64        = 25.0
    mu_psi::Float64       = 100.0
    sigma_psi::Float64    = 0.25
    sigma_x::Float64      = 0.5
    sigma_y::Float64      = 0.25
    decay_k::Float64      = 0.25
    K_x0::Float64         = 5.0
    K_y0::Float64         = 5.0
    K_psi0::Float64        = 20.0
end

@inline function smooth_bound(K, K_max)
    return 0.5 - 0.5 * tanh(1.0 * (K - (K_max - 2.0)))
end

@inline function get_dynamic_lambda(e, edot, lam_min, lam_max, mu)
    exp_term = exp(-mu * e^2)
    lam = lam_min + (lam_max - lam_min) * exp_term
    lam_dot = -2 * mu * e * edot * (lam_max - lam_min) * exp_term
    return lam, lam_dot
end

# asmc_wrench_at(bus, x_hat, ref, t, params, asmc, dt; use_dhat) -> SVector{3}
#
# Velocity-tracking adaptive SMC task-space wrench [Wx,Wy,Wpsi] (mirrors
# asmc_torques_vel, run_one.jl:538-649, minus the wheel-torque mapping and the
# super-twisting DOB -- the equivalent SMO d_hat feedforward is added instead
# when `use_dhat`). Integrates bus.K_asmc by forward-Euler at the ASMC rate.
# Only velocity-mode tracking is implemented (this notebook's HybridConfig demo
# runs use VelRef profiles per Profiles.jl's "ALL profiles are VelRef now"
# design); :pose mode is a documented extension point requiring a
# PosRef-tracking wrench, mirroring asmc_torques (run_one.jl:397-526).
function asmc_wrench_at(bus::ControllerBus, x_hat::AbstractVector, ref, t::Real, params::PlatformParams,
                        asmc::ASMCParams, dt::Real; use_dhat::Bool = false)
    Vx, Vy, psi_dot, psi = x_hat[1], x_hat[2], x_hat[3], x_hat[4]
    K_x, K_y, K_psi = bus.K_asmc[1], bus.K_asmc[2], bus.K_asmc[3]
    m, ms, Is, J1 = params.m, params.ms, params.Is, params.J_wheel
    aX, aY = params.aX, params.aY
    p1, p2 = params.p1_case1, params.p2_case1
    l, h, R, Rd = params.l, params.h, params.R, params.Ra
    I_psi = Is + 4*(l + h)^2 / R^2 * J1

    Vx_d, Vy_d, omega_d = ref.Vx(t), ref.Vy(t), ref.Wz(t)
    d_psi = psi - ref.psi(t)
    e_psi = 2 * tan(d_psi/2) * (1 + 2 * (1 - cos(d_psi)))
    edash_psi = (sec(d_psi/2))^2 * (3 - 2 * cos(d_psi)) + 4 * tan(d_psi/2) * sin(d_psi)
    edot_psi  = edash_psi * (psi_dot - omega_d)
    lam_psi, lam_dot_psi = get_dynamic_lambda(e_psi, edot_psi, asmc.lam_psi_min, asmc.lam_psi_max, asmc.mu_psi)

    s_x   = Vx - Vx_d
    s_y   = Vy - Vy_d
    s_psi = edot_psi + lam_psi * e_psi

    ss_x   = tanh(s_x   / asmc.eps)
    ss_y   = tanh(s_y   / asmc.eps)
    ss_psi = tanh(s_psi / asmc.eps_psi)

    Mx_sw   = -K_x   * ss_x
    My_sw   = -K_y   * ss_y
    Mpsi_sw = -K_psi * ss_psi

    Ax_eq    = ref.Ax(t)
    Ay_eq    = ref.Ay(t)
    alpha_eq = ref.al(t) - lam_dot_psi * e_psi - lam_psi * edot_psi * edash_psi

    Mx_eq = R * ((ms + 4*J1/R^2) * Ax_eq - ms * psi_dot * Vy - m * aY * alpha_eq
                 - m * aX * psi_dot^2 + 4 * p1 * Vx / R^2)
    My_eq = R * ((ms + 4*J1/R^2) * Ay_eq + ms * psi_dot * Vx + m * aX * alpha_eq
                 - m * aY * psi_dot^2 + (4 * p1 / R^2 + 8 * p2 / (R - Rd)^2) * Vy)
    M_psi_eq = (I_psi * alpha_eq - m * aY * (Ax_eq - psi_dot * Vy) + m * aX * (Ay_eq + psi_dot * Vx)
                + (4*p1*(l+h)^2/R^2 + 8*p2*h^2/(R-Rd)^2) * psi_dot)

    if use_dhat
        M_aug_dh = params.M_aug * bus.d_hat
        Mx_eq    -= R * M_aug_dh[1]
        My_eq    -= R * M_aug_dh[2]
        M_psi_eq -=     M_aug_dh[3]
    end

    base_dK_x   = asmc.gamma_x   * (s_x   * ss_x)
    base_dK_y   = asmc.gamma_y   * (s_y   * ss_y)
    base_dK_psi = asmc.gamma_psi * (s_psi * ss_psi)
    dK_x   = base_dK_x   * smooth_bound(K_x,   asmc.K_max_x)   - 0.1 * (K_x   / asmc.K_max_x)^3 - asmc.sigma_x   * (K_x   - asmc.K_x0   * 0.95) * exp(asmc.decay_k*(1 - s_x^2  /(9 * asmc.eps_psi^2)))
    dK_y   = base_dK_y   * smooth_bound(K_y,   asmc.K_max_y)   - 0.3 * (K_y   / asmc.K_max_y)^3 - asmc.sigma_y   * (K_y   - asmc.K_y0   * 0.95) * exp(asmc.decay_k*(1 - s_y^2  /(9 * asmc.eps_psi^2)))
    dK_psi = base_dK_psi * smooth_bound(K_psi, asmc.K_max_psi) - 0.5 * (K_psi / asmc.K_max_psi)^3 - asmc.sigma_psi * (K_psi - asmc.K_psi0 * 0.95) * exp(asmc.decay_k*(1 - s_psi^2/(9 * asmc.eps_psi^2)))
    bus.K_asmc = bus.K_asmc .+ dt .* SVector(dK_x, dK_y, dK_psi)

    bus.W_asmc = SVector(Mx_sw + Mx_eq, My_sw + My_eq, Mpsi_sw + M_psi_eq)
    return bus.W_asmc
end


In [ ]:
# --- Kalman-PID (Sec 4.6c) ---
Base.@kwdef struct PIDController
    Kp::Float64   = 40.0
    Ki::Float64   = 5.0
    Kd::Float64   = 0.5
    I_max::Float64 = 10.0
end

# pid_wrench!(bus, x_hat, ref, t, pid, dt) -> SVector{3}
#
# Velocity-error PID on the estimated body velocity, anti-windup saturation on
# the integral (paper eq. 32).
function pid_wrench!(bus::ControllerBus, x_hat::AbstractVector, ref, t::Real, pid::PIDController, dt::Real)
    Vx, Vy, psi_dot = x_hat[1], x_hat[2], x_hat[3]
    e = SVector(Vx - ref.Vx(t), Vy - ref.Vy(t), psi_dot - ref.Wz(t))

    I_new = bus.pid_integral .+ dt .* e
    I_new = clamp.(I_new, -pid.I_max, pid.I_max)      # anti-windup
    bus.pid_integral = I_new

    edot = e ./ dt      # simple backward-difference derivative on the (smooth) reference error
    W = -pid.Kp .* e .- pid.Ki .* I_new .- pid.Kd .* edot
    bus.W_pid = W
    return W
end


In [ ]:
# --- MPC (Sec 4.6b) ---
# Decision variable: per-wheel motor voltage U=[V1..V4] over a horizon Np, on a
# linear discrete body model x=[Vx,Vy,psi_dot] with B folding the motor map's
# quasi-static linearization (dTau/dV = G*eta*Kt/Ra) and the O-config
# allocation lever=R/(l+h), M_aug_inv. Cost: ||x-x_ref||^2_Q + ||U||^2_R +
# ||dU||^2_S (paper eqs. 18-23). Solved once per tick with OSQP, warm-started;
# only the first-step voltage is applied (receding horizon), then converted
# back to a task-space wrench so the mixer can blend it uniformly with ASMC/PID
# (Sec 4.6b: "if the mixer expects wrenches, return the equivalent task-space
# wrench").
Base.@kwdef struct MPCController
    Np::Int        = 10
    Q::SVector{3,Float64}  = SVector(50.0, 50.0, 80.0)
    R::SVector{4,Float64}  = SVector(0.01, 0.01, 0.01, 0.01)
    S::SVector{4,Float64}  = SVector(0.05, 0.05, 0.05, 0.05)
    rate_hz::Float64        = 100.0
end

@inline function mpc_body_matrices(params::PlatformParams, motor::MotorParams, dt::Real)
    R, l, h = params.R, params.l, params.h
    lever = R / (l + h)
    # O-config allocation: task wrench W=[Wx,Wy,Wpsi] -> 4 wheel torques (0.25 mix, run_one.jl:504-512).
    Amix = 0.25 .* SMatrix{4,3,Float64,12}(
        1.0, 1.0, 1.0, 1.0,
       -1.0, 1.0, 1.0,-1.0,
       -lever, lever, -lever, lever)
    # quasi-static voltage -> wheel-torque sensitivity at zero speed: dTau/dV = G*eta*Kt/Ra.
    dTaudV = motor.G * motor.eta * motor.Kt / motor.Ra
    # B_v: 4 wheel torques -> body acceleration, via the transpose allocation and M_aug_inv,
    # scaled by R for the force channels (mirrors run_one.jl's R*(...) equivalent-control form).
    Bwrench = SMatrix{3,4,Float64,12}(transpose(Amix) .* SVector(R, R, 1.0))
    A_body = SMatrix{3,3,Float64,9}(-dt .* Diagonal(SVector(4*params.p1_case1/R^2, 4*params.p1_case1/R^2, 0.0)) * params.M_aug_inv) + SMatrix{3,3,Float64,9}(I)
    B_body = dt .* params.M_aug_inv * Bwrench .* dTaudV
    return A_body, B_body
end

# mpc_wrench!(bus, x_hat, ref, t, params, motor, mpc) -> SVector{3}
#
# QP-MPC task-space wrench. Builds a linear discrete body model, solves a
# box-constrained (|V|<=V_max) least-squares tracking QP with OSQP over Np
# steps, applies the first-step voltage, converts back to a wrench via the
# same motor-torque sensitivity used to build B. Falls back to bus.mpc_last on
# solver failure/infeasibility (Sec 9 QP robustness).
function mpc_wrench!(bus::ControllerBus, x_hat::AbstractVector, ref, t::Real, params::PlatformParams,
                     motor::MotorParams, mpc::MPCController)
    dt = 1.0 / mpc.rate_hz
    A, B = mpc_body_matrices(params, motor, dt)
    Np = mpc.Np
    x0 = SVector(x_hat[1], x_hat[2], x_hat[3])

    # Stack the QP over the horizon: decision z = [U_1;...;U_Np] in R^{4*Np}.
    n, m = 3, 4
    Qd = Diagonal(repeat(mpc.Q, Np))
    Rd = Diagonal(repeat(mpc.R, Np))
    Sd = Diagonal(repeat(mpc.S, Np))

    # Prediction matrices: x_k = A^k x0 + sum_j A^(k-1-j) B u_j
    Sx = zeros(n*Np, n)
    Su = zeros(n*Np, m*Np)
    Ak = Matrix{Float64}(I, n, n)
    for k in 1:Np
        Ak = Ak * Matrix(A)
        Sx[(k-1)*n+1:k*n, :] = Ak
        Aj = Matrix{Float64}(I, n, n)
        for j in k:-1:1
            Su[(k-1)*n+1:k*n, (j-1)*m+1:j*m] = Aj * Matrix(B)
            Aj = Aj * Matrix(A)
        end
    end

    xref = vcat([SVector(ref.Vx(t + k*dt), ref.Vy(t + k*dt), ref.Wz(t + k*dt)) for k in 1:Np]...)

    # Rate-penalty difference operator D (Np*m x Np*m), D*u = [u1-u_prev; u2-u1; ...].
    D = zeros(m*Np, m*Np)
    for k in 1:Np
        D[(k-1)*m+1:k*m, (k-1)*m+1:k*m] .= Matrix{Float64}(I, m, m)
        if k > 1
            D[(k-1)*m+1:k*m, (k-2)*m+1:(k-1)*m] .= -Matrix{Float64}(I, m, m)
        end
    end
    u_prev_full = vcat(bus.mpc_last, zeros(m*(Np-1)))

    Hqp = Su' * Qd * Su + Rd + D' * Sd * D
    Hqp = (Hqp + Hqp') / 2 + 1e-6 * I
    fqp = Su' * Qd * (Sx * x0 .- xref) .- D' * Sd * (D * u_prev_full)

    lb = fill(-motor.V_max, m*Np)
    ub = fill( motor.V_max, m*Np)
    Aqp = sparse(1.0I, m*Np, m*Np)

    model = OSQP.Model()
    OSQP.setup!(model; P = sparse(Hqp), q = fqp, A = Aqp, l = lb, u = ub,
                verbose = false, warm_start = true, polish = true)
    results = OSQP.solve!(model)

    if results.info.status_val in (1, 2)   # SOLVED, SOLVED_INACCURATE
        u0 = SVector{4,Float64}(results.x[1:4])
        bus.mpc_last = u0
    else
        u0 = bus.mpc_last                   # fall back to previous command (Sec 9 QP robustness)
    end

    # Convert the applied voltage back to an equivalent task-space wrench so the
    # mixer can blend MPC uniformly with ASMC/PID (mixer re-derives voltage anyway).
    R_, l_, h_ = params.R, params.l, params.h
    lever = R_ / (l_ + h_)
    Amix = 0.25 .* SMatrix{4,3,Float64,12}(
        1.0, 1.0, 1.0, 1.0,
       -1.0, 1.0, 1.0,-1.0,
       -lever, lever, -lever, lever)
    dTaudV = motor.G * motor.eta * motor.Kt / motor.Ra
    tau_eq = dTaudV .* u0
    W = pinv(Matrix(Amix)) * Vector(tau_eq)
    Wv = SVector{3,Float64}(W)
    bus.W_mpc = Wv
    return Wv
end


## 9. `FuzzySupervisor` + `fuzzy_update!` (Sec 4.7)

In [ ]:
Base.@kwdef struct FuzzySupervisor
    ep_range::Tuple{Float64,Float64} = (0.0, 0.5)
    ev_range::Tuple{Float64,Float64} = (0.0, 0.3)
    edp_range::Tuple{Float64,Float64} = (-0.2, 0.2)
end

@inline function tri_mf(x, a, b, c)
    x <= a && return 0.0
    x >= c && return 0.0
    x <= b && return (x - a) / (b - a)
    return (c - x) / (c - b)
end

# Type-1 triangular MFs over {SMALL, MEDIUM, LARGE} on each of (ep, ev, edp) (paper Sec 3.4).
@inline function mf_triplet(x, lo, hi)
    mid = (lo + hi) / 2
    small  = tri_mf(x, lo - (hi-lo), lo, mid)
    medium = tri_mf(x, lo, mid, hi)
    large   = tri_mf(x, mid, hi, hi + (hi-lo))
    s = small + medium + large + 1e-9
    return small/s, medium/s, large/s
end

# fuzzy_update!(bus, x_hat, ref, t, cfg, fz) -> SVector{3}
#
# Sets bus.weights = (w_ASMC, w_MPC, w_PID), normalized over the ENABLED
# controllers, from (e_p, e_v, edot_p) features on the estimated velocity
# error. Reduces to the paper's single beta(k) (MPC<->PID) when only MPC,PID
# are enabled. If cfg.fuzzy=false, uses cfg.fixed_weights.
function fuzzy_update!(bus::ControllerBus, x_hat::AbstractVector, ref, t::Real, cfg::HybridConfig, fz::FuzzySupervisor)
    if !cfg.fuzzy
        w = SVector(cfg.fixed_weights)
    else
        e_v = sqrt((x_hat[1]-ref.Vx(t))^2 + (x_hat[2]-ref.Vy(t))^2)
        e_p = e_v                                    # velocity-mode: position-like feature = velocity-error magnitude
        e_dp = abs(x_hat[3] - ref.Wz(t))

        s1,m1,l1 = mf_triplet(clamp(e_p,  fz.ep_range...),  fz.ep_range...)
        s2,m2,l2 = mf_triplet(clamp(e_v,  fz.ev_range...),  fz.ev_range...)
        s3,m3,l3 = mf_triplet(clamp(e_dp, fz.edp_range...), fz.edp_range...)

        # Singleton output rule base: LARGE error -> favor MPC (predictive, constraint-aware);
        # SMALL error -> favor PID (cheap, well-damped near setpoint); ASMC scales with
        # combined error magnitude throughout (robust to unmodeled disturbance/slip).
        agg_small = (s1 + s2 + s3) / 3
        agg_large = (l1 + l2 + l3) / 3
        agg_med   = 1.0 - agg_small - agg_large

        w_pid  = agg_small
        w_mpc  = agg_large
        w_asmc = agg_med + 0.25*(agg_small + agg_large)   # ASMC always carries some authority

        w = SVector(w_asmc, w_mpc, w_pid)
    end

    mask = SVector(cfg.use_asmc ? 1.0 : 0.0, cfg.use_mpc ? 1.0 : 0.0, cfg.use_pid ? 1.0 : 0.0)
    w = w .* mask
    total = sum(w)
    w = total > 1e-9 ? w ./ total : mask ./ max(sum(mask), 1.0)

    bus.weights = w
    return w
end


## 10. `mix_and_allocate!` (Sec 4.8)

In [ ]:
# mix_and_allocate!(bus, params, motor, cfg, dt) -> SVector{4}
#
# Blends enabled wrenches by bus.weights -> W; maps W to 4 wheel torques via
# the O-config allocation (lever=R/(l+h), 0.25 mix; run_one.jl:504-512/627-635);
# inverts the youBot motor map to per-wheel voltage at the current omega
# (estimated wheel speed from the encoder measurement cache); saturates to
# +-V_max with |i|<=i_max and a per-tick dV rate limit; writes bus.v_cmd.
function mix_and_allocate!(bus::ControllerBus, params::PlatformParams, motor::MotorParams, cfg::HybridConfig, dt::Real)
    W = bus.weights[1] .* bus.W_asmc .+ bus.weights[2] .* bus.W_mpc .+ bus.weights[3] .* bus.W_pid

    R, l, h = params.R, params.l, params.h
    lever = R / (l + h)
    Mx, My, Mpsi = W[1], W[2], W[3]
    tau1 = 0.25 * (Mx - My - lever * Mpsi)
    tau2 = 0.25 * (Mx + My + lever * Mpsi)
    tau3 = 0.25 * (Mx + My - lever * Mpsi)
    tau4 = 0.25 * (Mx - My + lever * Mpsi)
    tau  = SVector(tau1, tau2, tau3, tau4)
    tau_sat = smooth_sat.(tau, params.Max_torque, 4)

    # Invert the youBot motor map at the measured wheel speed: V = i*Ra + Kb*G*omega,
    # i = tau_wheel / (G*eta*Kt) (+ tau_f/(G*eta) shaft-friction correction).
    omega_meas = bus.y_meas.omega
    omega_m = motor.G .* omega_meas
    tau_f_smooth = motor.tau_f .* tanh.(20.0 .* omega_m)
    i_cmd = (tau_sat .+ motor.G * motor.eta .* tau_f_smooth) ./ (motor.G * motor.eta * motor.Kt)
    i_cmd = smooth_sat.(i_cmd, motor.i_max, 3)
    v_raw = i_cmd .* motor.Ra .+ motor.Kb .* omega_m

    v_prev = bus.v_cmd
    dV_max = 200.0 * dt          # per-tick voltage rate limit [V/s] * dt
    dv = clamp.(v_raw .- v_prev, -dV_max, dV_max)
    v_cmd = smooth_sat.(v_prev .+ dv, motor.V_max, 3)

    bus.v_cmd = v_cmd
    return v_cmd
end


## 11. `build_callbacks` / `run_hybrid` (Sec 4.11)

In [ ]:
# Assembled composite callback ordering (Sec 9 "Callback ordering"): within a
# shared tick, the update order is estimator -> ASMC -> MPC -> PID -> fuzzy ->
# mixer, all driven off a single fastest-rate PeriodicCallback (f_mix, >= every
# other rate) with internal counters gating the slower blocks -- this avoids
# the stale-x_hat/weights race a set of independently-scheduled
# PeriodicCallbacks would have.
mutable struct HybridRuntime
    cfg::HybridConfig
    params::PlatformParams
    motor::MotorParams
    sm::SensorModel
    sstate::SensorState
    kf::Union{KalmanEstimator,Nothing}
    smo::Union{SMOEstimator,Nothing}
    asmc::ASMCParams
    pid::PIDController
    mpc::MPCController
    fz::FuzzySupervisor
    ref::Any
    tick::Int
    log::Vector{NamedTuple}
end

function build_runtime(cfg::HybridConfig, params::PlatformParams, motor::MotorParams, ref)
    sm = SensorModel(seed = cfg.seed)
    HybridRuntime(cfg, params, motor, sm, SensorState(sm),
                  cfg.estimator == :kalman ? KalmanEstimator() : nothing,
                  cfg.estimator == :smo    ? SMOEstimator()    : nothing,
                  ASMCParams(), PIDController(), MPCController(), FuzzySupervisor(),
                  ref, 0, NamedTuple[])
end

function master_tick!(integrator)
    rt::HybridRuntime = integrator.p.rt_ref[]
    bus = integrator.p.bus
    cfg = rt.cfg
    t = integrator.t
    dt_mix = 1.0 / cfg.f_mix
    rt.tick += 1

    n_est  = max(1, round(Int, cfg.f_mix / cfg.f_est))
    n_asmc = max(1, round(Int, cfg.f_mix / cfg.f_asmc))
    n_mpc  = max(1, round(Int, cfg.f_mix / cfg.f_mpc))
    n_pid  = max(1, round(Int, cfg.f_mix / cfg.f_pid))
    n_fuzz = max(1, round(Int, cfg.f_mix / cfg.f_fuzzy))

    du = similar(integrator.u)
    plant_rhs!(du, integrator.u, integrator.p, t)

    if cfg.estimator != :none && rt.tick % n_est == 0
        y = simulate_measurement(integrator.u, du, rt.sm, rt.sstate, n_est*dt_mix)
        bus.y_meas = y
        est = cfg.estimator == :kalman ? rt.kf : rt.smo
        estimator_update!(bus, y, est, rt.params, n_est*dt_mix)
        if !(cfg.estimator == :smo && cfg.use_dhat)
            bus.d_hat = SVector(0.0,0.0,0.0)
        end
    end

    x_hat = bus.x_hat
    if cfg.use_asmc && rt.tick % n_asmc == 0
        asmc_wrench_at(bus, x_hat, rt.ref, t, rt.params, rt.asmc, n_asmc*dt_mix; use_dhat = cfg.use_dhat)
    end
    if cfg.use_mpc && rt.tick % n_mpc == 0
        mpc_wrench!(bus, x_hat, rt.ref, t, rt.params, rt.motor, rt.mpc)
    end
    if cfg.use_pid && rt.tick % n_pid == 0
        pid_wrench!(bus, x_hat, rt.ref, t, rt.pid, n_pid*dt_mix)
    end
    if rt.tick % n_fuzz == 0
        fuzzy_update!(bus, x_hat, rt.ref, t, cfg, rt.fz)
    end

    mix_and_allocate!(bus, rt.params, rt.motor, cfg, dt_mix)

    push!(rt.log, (t = t, u = copy(integrator.u), x_hat = bus.x_hat, v_cmd = bus.v_cmd,
                   W_asmc = bus.W_asmc, W_mpc = bus.W_mpc, W_pid = bus.W_pid,
                   weights = bus.weights, tau_applied = bus.tau_wheel_applied,
                   d_hat = bus.d_hat))
    return nothing
end

# PlantODEParams doesn't carry `rt`; wrap it so the callback can reach the
# runtime without changing plant_rhs!'s signature (rt_ref is a Ref{Any} box).
struct HybridODEParams
    params::PlatformParams
    chi::Float64
    p1::Float64
    p2::Float64
    coupling::Symbol
    lugre::LuGreParams
    motor::MotorParams
    bus::ControllerBus
    rt_ref::Base.RefValue{Any}
end
plant_rhs!(du, u, p::HybridODEParams, t) = plant_rhs!(du, u,
    PlantODEParams(p.params, p.chi, p.p1, p.p2, p.coupling, p.lugre, p.motor, p.bus), t)

# run_hybrid(cfg, params, motor; chi, friction_case, lugre, config_dir, profile_set) -> (sol, rt)
#
# Build u0 + bus + ODEProblem, install the config-gated master
# PeriodicCallback, solve with a stiff solver + ref.tstops, return
# (sol, HybridRuntime) -- rt.log holds the per-tick controller/estimator
# record for log_run.
function run_hybrid(cfg::HybridConfig, params::PlatformParams, motor::MotorParams;
                    chi::Real, friction_case::Int = 1, lugre::LuGreParams = lugre,
                    config_dir::AbstractString = ".",
                    profile_set::AbstractVector{<:AbstractString} = ["profiles/octagon.toml"])
    ref, _, _ = Profiles.pick_and_build(config_dir, profile_set)
    p1, p2 = friction_case == 1 ? (params.p1_case1, params.p2_case1) : (params.p1_case2, params.p2_case2)

    u0 = build_initial_state(cfg)
    bus = ControllerBus()
    rt = build_runtime(cfg, params, motor, ref)
    rt_ref = Base.RefValue{Any}(rt)
    p = HybridODEParams(params, Float64(chi), p1, p2, coupling_of(:lugre_adamov), lugre, motor, bus, rt_ref)

    T = ref.T_total
    prob = ODEProblem(plant_rhs!, u0, (0.0, T), p)
    dt_mix = 1.0 / cfg.f_mix
    mastercb = PeriodicCallback(master_tick!, dt_mix)

    sol = solve(prob, TRBDF2();
                reltol = cfg.reltol, abstol = ABSTOL_HYBRID[1:length(u0)],
                tstops = ref.tstops, callback = mastercb, dtmax = cfg.dtmax, maxiters = 10^7)
    return sol, rt
end


## 12. `log_run` / `save_run` (Sec 4.12)

In [ ]:
# log_run/save_run assemble the per-tick controller log into a DataFrame and
# persist it via Arrow -- an ADDITIVE schema next to the existing DataStore
# convention (see the notebook-header note): true state, estimated state,
# per-controller wrenches, blend weights, applied wheel torques and voltage
# command over the run.
function log_run(sol, rt::HybridRuntime)
    recs = rt.log
    N = length(recs)
    df = DataFrame(
        t = [r.t for r in recs],
        Vx = [r.u[1] for r in recs], Vy = [r.u[2] for r in recs],
        psi_dot = [r.u[3] for r in recs], psi = [r.u[4] for r in recs],
        Xo = [r.u[17] for r in recs], Yo = [r.u[18] for r in recs],
        Vx_hat = [r.x_hat[1] for r in recs], Vy_hat = [r.x_hat[2] for r in recs],
        psi_dot_hat = [r.x_hat[3] for r in recs], psi_hat = [r.x_hat[4] for r in recs],
        Xo_hat = [r.x_hat[5] for r in recs], Yo_hat = [r.x_hat[6] for r in recs],
        v1 = [r.v_cmd[1] for r in recs], v2 = [r.v_cmd[2] for r in recs],
        v3 = [r.v_cmd[3] for r in recs], v4 = [r.v_cmd[4] for r in recs],
        tau1 = [r.tau_applied[1] for r in recs], tau2 = [r.tau_applied[2] for r in recs],
        tau3 = [r.tau_applied[3] for r in recs], tau4 = [r.tau_applied[4] for r in recs],
        w_asmc = [r.weights[1] for r in recs], w_mpc = [r.weights[2] for r in recs],
        w_pid = [r.weights[3] for r in recs],
        Wx_asmc = [r.W_asmc[1] for r in recs], Wy_asmc = [r.W_asmc[2] for r in recs], Wpsi_asmc = [r.W_asmc[3] for r in recs],
        Wx_mpc  = [r.W_mpc[1]  for r in recs], Wy_mpc  = [r.W_mpc[2]  for r in recs], Wpsi_mpc  = [r.W_mpc[3]  for r in recs],
        Wx_pid  = [r.W_pid[1]  for r in recs], Wy_pid  = [r.W_pid[2]  for r in recs], Wpsi_pid  = [r.W_pid[3]  for r in recs],
        d_hat_x = [r.d_hat[1] for r in recs], d_hat_y = [r.d_hat[2] for r in recs], d_hat_psi = [r.d_hat[3] for r in recs],
    )
    return df
end

function save_run(df::DataFrame, outdir::AbstractString, tag::AbstractString)
    isdir(outdir) || mkpath(outdir)
    path = joinpath(outdir, "hybrid_" * tag * ".arrow")
    Arrow.write(path, df; compress = :zstd)
    return path
end


## 13. Demo run

Runs one closed-loop trajectory with the default `HybridConfig` (ASMC+MPC+PID,
fuzzy blend, Kalman estimator, velocity tracking) and plots tracking + blend
weights. Requires a `config_dir` with a `profiles/octagon.toml` (any of the
project's existing `trajectory_files_run_*` directories work) -- point
`CONFIG_DIR` at one before running.

In [ ]:
CONFIG_DIR = "trajectory_files_run_0p_main"   # <- point at an existing trajectory config dir
PROFILE_SET = ["profiles/octagon.toml"]

demo_params = PlatformParams(Profiles.load_base(CONFIG_DIR); mu_friction = 0.5)
demo_motor  = MotorParams()

sol, rt = run_hybrid(cfg, demo_params, demo_motor; chi = 0.002, friction_case = 1,
                     config_dir = CONFIG_DIR, profile_set = PROFILE_SET)
df = log_run(sol, rt)

plt1 = plot(df.t, [df.Vx df.Vx_hat]; label=["Vx true" "Vx hat"], xlabel="t [s]", ylabel="Vx [m/s]")
plt2 = plot(df.t, [df.Vy df.Vy_hat]; label=["Vy true" "Vy hat"], xlabel="t [s]", ylabel="Vy [m/s]")
plt3 = plot(df.t, [df.w_asmc df.w_mpc df.w_pid]; label=["w_ASMC" "w_MPC" "w_PID"], xlabel="t [s]", ylabel="blend weight")
plt4 = plot(df.t, [df.v1 df.v2 df.v3 df.v4]; label=["V1" "V2" "V3" "V4"], xlabel="t [s]", ylabel="motor voltage [V]")
plot(plt1, plt2, plt3, plt4; layout=(2,2), size=(1000,700))


## 14. Selection-matrix comparison (Sec 10 success criteria)

Loops the enabled-controller x fuzzy x estimator selection matrix; each config
must execute without error and log weights consistent with the enabled set.

In [ ]:
selection_matrix = HybridConfig[]
for (ua, um, up) in [(true,false,false), (false,true,false), (false,false,true), (true,true,true)]
    for fuzzy in (false, true)
        for est in (:kalman, :smo)
            push!(selection_matrix, HybridConfig(use_asmc=ua, use_mpc=um, use_pid=up,
                                                  fuzzy=fuzzy, estimator=est, tracking=:velocity))
        end
    end
end
println("Selection matrix size: ", length(selection_matrix))

results = NamedTuple[]
@showprogress for (i, c) in enumerate(selection_matrix)
    local sol_i, rt_i
    try
        sol_i, rt_i = run_hybrid(c, demo_params, demo_motor; chi = 0.002, friction_case = 1,
                                 config_dir = CONFIG_DIR, profile_set = PROFILE_SET)
        df_i = log_run(sol_i, rt_i)
        vel_err = sqrt(sum((df_i.Vx .- df_i.Vx_hat).^2 .+ (df_i.Vy .- df_i.Vy_hat).^2) / nrow(df_i))
        push!(results, (idx=i, use_asmc=c.use_asmc, use_mpc=c.use_mpc, use_pid=c.use_pid,
                        fuzzy=c.fuzzy, estimator=c.estimator, ok=true, vel_rmse=vel_err))
    catch e
        @warn "selection matrix run $i failed" exception=e
        push!(results, (idx=i, use_asmc=c.use_asmc, use_mpc=c.use_mpc, use_pid=c.use_pid,
                        fuzzy=c.fuzzy, estimator=c.estimator, ok=false, vel_rmse=NaN))
    end
end
DataFrame(results)


## 15. Save

In [ ]:
save_run(df, "runs_hybrid", "demo_octagon_asmc_mpc_pid_fuzzy")
